# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

## 1. My lane as an ML task (type)

**Task type: Classification** (binary).

Lane 1 (Ranking Signal Analysis) is fundamentally about flagging *which*
pages are losing visibility so a content team can prioritize review. I'm
framing this as binary classification: predict whether a page is
**declining** or **not declining**, using signals like `avg_position`,
`ctr`, `freshness_tier`, and `word_count_tier` as inputs. The model's
predicted probability doubles as a priority score — pages can be sorted by
"how likely is this page declining" to build a review shortlist, which is
exactly the action the content team needs.

In [1]:
import os
import pandas as pd

REPO_NAME = "FlyRankAI_ML-Engineering_Internship"
REPO_URL = "https://github.com/Sadman-Islam-Chowdhury-Samin/FlyRankAI_ML-Engineering_Internship.git"

if os.getcwd().split("/")[-1] != REPO_NAME:
    if not os.path.exists(f"/content/{REPO_NAME}"):
        !git clone {REPO_URL} /content/{REPO_NAME}
    os.chdir(f"/content/{REPO_NAME}")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Working directory:", os.getcwd())
print("Shape:", df.shape)
print("Class balance (trend_direction):")
print(df["trend_direction"].value_counts(normalize=True))

Working directory: /content/FlyRankAI_ML-Engineering_Internship
Shape: (30000, 44)
Class balance (trend_direction):
trend_direction
down      0.542067
stable    0.198733
up        0.146267
new       0.074533
flat      0.038400
Name: proportion, dtype: float64


Note: `trend_direction` has five categories (down, stable, up, new, flat),
not two. Grouping everything except "down" into a single "not declining"
class is a simplification — it treats "new" pages (no trend history yet)
the same as genuinely "stable" or "up" pages. This is a reasonable first
pass for a binary classifier, but a more careful version later might
exclude "new" pages entirely or model this as multi-class instead.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

## 2. Target or proxy

**Target:** `is_declining` — a binary label I'll derive as
`1 if trend_direction == "down" else 0`.

This isn't an invented rule — `trend_direction` itself is already an
observed outcome in the dataset, computed by comparing
`impressions_last_30d` against `impressions_prev_30d`. I'm just converting
it into a clean 0/1 label for modeling. Because the label comes directly
from `trend_direction`/`trend_pct`, those two columns can never be used as
input features — that would be leaking the answer into the inputs.

In [2]:
df["is_declining"] = (df["trend_direction"] == "down").astype(int)
print(df["is_declining"].value_counts())
print("Share declining:", df["is_declining"].mean())

is_declining
1    16262
0    13738
Name: count, dtype: int64
Share declining: 0.5420666666666667


## 3. Success metric

*One metric you can defend. What number means 'good'?*

## 3. Success metric

**Metric: Precision@top-20%** — of the pages the model flags as highest-risk
(top 20% by predicted probability of declining), what share are actually
declining?

I'm not using plain accuracy, since ~54% of pages are already declining —
a model that just guesses "declining" for everyone would look deceptively
accurate. I'm also not leading with ROC-AUC, since it measures overall
ranking quality but doesn't tell me what the content team actually
experiences: they won't review all 30,000 pages, they'll review a shortlist.
Precision@top-20% directly answers "if I only have time to review the
top-flagged pages, how many of them are worth reviewing?" — which ties the
metric straight to the real action.

In [3]:
baseline_precision = df["is_declining"].mean()
print("Baseline (random 20% sample) expected precision:", round(baseline_precision, 3))
print("Any model needs to beat this baseline meaningfully to be useful.")

Baseline (random 20% sample) expected precision: 0.542
Any model needs to beat this baseline meaningfully to be useful.


**Action this supports:** A content strategist takes the model's output —
pages ranked by predicted probability of decline — and reviews the
top-20%-flagged pages first for refresh, rewrite, or deeper investigation,
instead of reviewing pages in publish-date order or waiting for manual
spot-checks.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

## 4. The unit of analysis, as a real dataframe

One row = **one content page** (`content_id`). Below is a real slice showing
the unit of analysis: the candidate feature columns plus the target.

In [4]:
feature_cols = [
    "content_id", "avg_position", "ctr", "freshness_tier",
    "word_count_tier", "content_type", "main_intent",
    "impressions_90d", "engagement_rate", "is_declining"
]
df[feature_cols].head(10)

,content_id,avg_position,ctr,freshness_tier,word_count_tier,content_type,main_intent,impressions_90d,engagement_rate,is_declining
0,content_304f48230142,10.6,0.76,0-30,2000-3500,keyword article,transactional,3803,5.88,1
1,content_a1fb4e703a9e,20.3,0.05,0-30,2000-3500,keyword article,informational,15320,0.00,1
2,content_9aa793d4d895,36.5,0.09,0-30,3500+,keyword article,informational,12581,0.00,1
3,content_331d6c4de07b,6.2,0.49,0-30,NaN,keyword article,commercial,11751,1.28,0
4,content_d99b7a2d90ca,44.0,0.13,0-30,2000-3500,keyword article,informational,19140,0.00,1
5,content_d4084a4bc775,8.5,0.03,0-30,2000-3500,keyword article,transactional,3970,0.00,1
6,content_9a34b442b552,7.0,0.00,0-30,2000-3500,keyword article,informational,20,0.00,1
7,content_a63219c6e95a,21.2,0.06,0-30,NaN,keyword article,commercial,1724,3.57,0
8,content_5e6c160719bc,46.0,0.09,0-30,3500+,keyword article,informational,32574,5.88,1
9,content_c27558df2b0c,4.9,0.16,91-180,NaN,keyword article,informational,1240,0.00,1


Note: some rows have `NaN` in `word_count_tier` (visible above) — missing
word count data for those pages. This will need handling (drop, impute, or
treat as its own category) before any model can use this column as a
feature.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

## 5. Why ML beats a fixed rule here

A fixed rule (e.g. "flag any page older than 180 days" or "flag any page
with position worse than 15") doesn't hold up. Checking individual
correlations with `is_declining` directly: `avg_position` (-0.029), `ctr`
(-0.062), `word_count` (0.090), `impressions_90d` (-0.018), and
`engagement_rate` (-0.013) are all weak — the strongest is `word_count` at
just 0.09. No single signal moves with decline strongly enough to write an
if-statement around; each one alone barely separates declining pages from
stable ones.

This is also consistent with last week's finding that `engagement_rate` by
`freshness_tier` was non-monotonic rather than a clean trend. Individually
weak but potentially complementary signals are exactly what a model (even a
simple logistic regression or tree) is for — combining several noisy inputs
into one score is likely to outperform any single-column rule, which is why
this is a modeling problem, not a fixed rule.

In [5]:
check_cols = ["avg_position", "ctr", "word_count", "impressions_90d", "engagement_rate"]
print("Individual correlation with is_declining:")
print(df[check_cols + ["is_declining"]].corr()["is_declining"].drop("is_declining"))

Individual correlation with is_declining:
avg_position      -0.029035
ctr               -0.061911
word_count         0.090157
impressions_90d   -0.018175
engagement_rate   -0.012743
Name: is_declining, dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [ ✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅ ] No client names, URLs, or private queries anywhere
- [ ✅] My claims use careful words: observed, measured, directional, decision-support
- [ ✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.